# TP1 SQLite mini : Northwind à la main

Pour le TP1, on n'a besoin que de la table `customers`. On la définit directement en Python (liste de dicts, vraies données extraites du dump Northwind), on l'insère dans SQLite en mémoire, et on fait tourner les requêtes du chapitre 1.

À la fin du notebook, une méthode alternative pour charger le Northwind complet à partir du fichier `data/northwind.sql` (en 4 lignes), si on veut s'entraîner sur les 91 clients.

## 1. Connexion

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

## 2. Les données (10 vrais clients Northwind)

Liste de dicts. On garde 6 colonnes : `customer_id`, `company_name`, `contact_name`, `city`, `country`, `region`. Les `None` (region) servent à tester `IS NULL`.

In [ ]:
customers = [
    {"customer_id": "ALFKI", "company_name": "Alfreds Futterkiste",        "contact_name": "Maria Anders",      "city": "Berlin",     "country": "Germany",   "region": None},
    {"customer_id": "ANATR", "company_name": "Ana Trujillo Emparedados",   "contact_name": "Ana Trujillo",      "city": "México D.F.","country": "Mexico",    "region": None},
    {"customer_id": "AROUT", "company_name": "Around the Horn",            "contact_name": "Thomas Hardy",      "city": "London",     "country": "UK",        "region": None},
    {"customer_id": "BERGS", "company_name": "Berglunds snabbköp",         "contact_name": "Christina Berglund","city": "Luleå",      "country": "Sweden",    "region": None},
    {"customer_id": "BLONP", "company_name": "Blondesddsl père et fils",   "contact_name": "Frédérique Citeaux","city": "Strasbourg", "country": "France",    "region": None},
    {"customer_id": "BONAP", "company_name": "Bon app'",                   "contact_name": "Laurence Lebihan",  "city": "Marseille",  "country": "France",    "region": None},
    {"customer_id": "BOTTM", "company_name": "Bottom-Dollar Markets",      "contact_name": "Elizabeth Lincoln", "city": "Tsawassen",  "country": "Canada",    "region": "BC"},
    {"customer_id": "GREAL", "company_name": "Great Lakes Food Market",    "contact_name": "Howard Snyder",     "city": "Eugene",     "country": "USA",       "region": "OR"},
    {"customer_id": "HANAR", "company_name": "Hanari Carnes",              "contact_name": "Mario Pontes",      "city": "Rio de Janeiro","country": "Brazil", "region": "RJ"},
    {"customer_id": "HILAA", "company_name": "HILARION-Abastos",           "contact_name": "Carlos Hernández",  "city": "San Cristóbal","country": "Venezuela","region": "Táchira"},
]
print(len(customers), "clients")

## 3. Créer la table et insérer

Placeholders nommés `:cle` : ils matchent directement les clés des dicts.

In [ ]:
cur.execute("""
CREATE TABLE customers (
    customer_id   TEXT PRIMARY KEY,
    company_name  TEXT,
    contact_name  TEXT,
    city          TEXT,
    country       TEXT,
    region        TEXT
)
""")

cur.executemany(
    """INSERT INTO customers
       VALUES (:customer_id, :company_name, :contact_name, :city, :country, :region)""",
    customers,
)
conn.commit()

## 4. Helper d'affichage

Petit utilitaire pour imprimer lisiblement (sans pandas).

In [ ]:
def show(sql, params=()):
    rows = cur.execute(sql, params).fetchall()
    cols = [d[0] for d in cur.description]
    widths = [max(len(c), *(len(str(r[i])) for r in rows)) if rows else len(c) for i, c in enumerate(cols)]
    fmt = " | ".join(f"{{:<{w}}}" for w in widths)
    print(fmt.format(*cols))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(fmt.format(*(str(v) if v is not None else "NULL" for v in r)))

## 5. SELECT

In [ ]:
show("SELECT contact_name, city FROM customers")

## 6. ORDER BY

In [ ]:
show("SELECT contact_name, country FROM customers ORDER BY country, contact_name")

## 7. DISTINCT

In [ ]:
show("SELECT DISTINCT country FROM customers ORDER BY country")

## 8. WHERE

In [ ]:
show("SELECT contact_name, city FROM customers WHERE country = 'France'")

Avec un paramètre (à préférer à la concaténation, pour éviter les injections SQL) :

In [ ]:
show("SELECT contact_name, city FROM customers WHERE country = ?", ("USA",))

## 9. LIMIT

In [ ]:
show("SELECT contact_name FROM customers ORDER BY contact_name LIMIT 3")

## 10. IS NULL

In [ ]:
show("SELECT customer_id, contact_name, region FROM customers WHERE region IS NULL")

Et l'inverse :

In [ ]:
show("SELECT customer_id, contact_name, region FROM customers WHERE region IS NOT NULL")

## Bonus : charger le Northwind complet depuis le dump

Si on veut les 91 clients (et toutes les autres tables) plutôt que cet extrait à 10 lignes, voici les 4 lignes nécessaires pour adapter le dump PostgreSQL à SQLite. Décommenter pour exécuter.

Une fois lancée, on a accès à toutes les tables : `customers` (91 lignes), `orders` (830), `products` (77), `employees` (9), etc.

In [ ]:
# import re
#
# with open("../../../data/northwind.sql") as f:
#     sql = f.read()
# sql = re.sub(r"^SET .*?;\s*$", "", sql, flags=re.M)
# sql = re.sub(r"ALTER TABLE ONLY[^;]*;", "", sql, flags=re.S)
# sql = sql.replace("bytea", "BLOB").replace(r"'\x'", "NULL")
#
# # nouvelle connexion pour ne pas écraser celle du dessus
# conn = sqlite3.connect(":memory:")
# cur = conn.cursor()
# conn.executescript(sql)
# conn.commit()
# show("SELECT COUNT(*) AS n FROM customers")

## Fermeture

In [ ]:
conn.close()